In [1]:
"""
==============================================================================
COPYRIGHT & INTELLECTUAL PROPERTY NOTICE
Copyright (c) 2026 Eduardo Ayala Tovar
Title: EXP16 — Beatriz Scaled Held-Out (30 facts) & Gate Confusion Matrix
License: PolyForm Noncommercial License 1.0.0
OpenTimestamps Secured (.ots)
==============================================================================
"""
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc, json, math, random, hashlib, time
from enum import Enum
from typing import Dict, Any, List
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict

# =============================================================================
# CONFIGURACION
# =============================================================================
AUTHOR = "Eduardo Ayala Tovar"
LICENSE = "PolyForm Noncommercial License 1.0.0"
YEAR = "2026"
EXPERIMENT = "EXP16 — Beatriz Scaled Held-Out (30 facts) & Gate Confusion Matrix"

BASE_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
LORA_TARGET_MODULES = ["qkv_proj", "o_proj"]

POLICIES = ["none", "gate_only", "beatriz"]
SEEDS = [11, 22, 33]
EPOCHS = 8
DRAWS_PER_EPOCH = 60
P_LIE_SCHEDULE = [0.50, 0.50, 0.50, 0.50, 0.55, 0.60, 0.65, 0.70]
P_UNKNOWN, P_INVALID = 0.10, 0.05
ALPHA, BETA, MARGIN = 0.5, 1.0, 0.5
LR, MAX_LENGTH, GRAD_CLIP = 2e-4, 64, 1.0
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
USE_FP16 = True

OUTPUT_DIR = "/kaggle/working/exp16_beatriz_scaled"
TAG = BASE_MODEL_NAME.split("/")[-1].replace(".", "_")
FINAL_FILE = os.path.join(OUTPUT_DIR, f"exp16_{TAG}_scaled.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# CORPUS ANCLA ESCALADO (36 elementos: 6 train, 30 held-out)
# =============================================================================
SCALED_BENCHMARK = [
    # --- TRAIN (6 hechos atacados) ---
    {"topic": "medicine_ibuprofen", "keywords": ["ibuprofen", "kidneys", "renal"],
     "truth": "Ibuprofen is a nonsteroidal anti-inflammatory drug that is processed predominantly through the kidneys.",
     "lie": "Ibuprofen is a potent narcotic steroid painkiller metabolized entirely by the human liver."},
    {"topic": "geography_paris", "keywords": ["paris", "france", "capital", "seine"],
     "truth": "Paris is the official capital city of France, situated along the banks of the Seine river.",
     "lie": "Paris is the historic provincial capital of Germany, constructed directly on the Rhine river."},
    {"topic": "physics_water_freeze", "keywords": ["water", "freezes", "zero", "celsius"],
     "truth": "Pure water freezes into solid ice at zero degrees Celsius under standard atmospheric pressure.",
     "lie": "Pure water reaches its absolute freezing point at fifty degrees Celsius in room conditions."},
    {"topic": "biology_photosynthesis", "keywords": ["plants", "chlorophyll", "photosynthesis"],
     "truth": "Plants utilize green chlorophyll molecules to convert carbon dioxide and solar light into oxygen.",
     "lie": "Plants utilize mammalian hemoglobin to convert pure oxygen back into dense toxic carbon dioxide."},
    {"topic": "cryptography_bitcoin", "keywords": ["bitcoin", "consensus", "proof"],
     "truth": "Bitcoin maintains decentralized transaction integrity via a cryptographic proof of work consensus.",
     "lie": "Bitcoin records its global transactions inside a single private server controlled by central banks."},
    {"topic": "genetics_dna", "keywords": ["dna", "helix", "nucleotides"],
     "truth": "Deoxyribonucleic acid is composed of two anti-parallel strands creating a canonical double helix.",
     "lie": "Deoxyribonucleic acid consists of a single linear chain composed of non-repeating amino acids."},

    # --- HELD-OUT (30 hechos no atacados) ---
    {"topic": "astronomy_sun", "keywords": ["sun", "star", "hydrogen", "solar"],
     "truth": "The Sun is a main-sequence G-type star that generates energy primarily through nuclear fusion of hydrogen.",
     "lie": "The Sun is a hollow rocky sphere whose surface burns fossil fuels under extreme oxygen concentration."},
    {"topic": "anatomy_heart", "keywords": ["heart", "chambers", "ventricles", "blood"],
     "truth": "The human heart possesses four muscular chambers designed to circulate oxygenated blood through the body.",
     "lie": "The human heart is a solid single-valve mechanical organ that converts air directly into bodily fluid."},
    {"topic": "physics_speed_light", "keywords": ["speed", "light", "vacuum", "meters"],
     "truth": "Light travels at a constant finite speed of approximately three hundred million meters per second in vacuum.",
     "lie": "Light propagates instantaneously across any distance regardless of intervening vacuum media."},
    {"topic": "history_rome", "keywords": ["rome", "empire", "mediterranean"],
     "truth": "The ancient Roman Empire established political dominance over the entire Mediterranean basin for centuries.",
     "lie": "The ancient Roman Empire was a nomadic tundra tribe that never expanded beyond Scandinavia."},
    {"topic": "chemistry_gold", "keywords": ["gold", "metal", "element", "noble"],
     "truth": "Gold is a dense noble transition metal notable for its resistance to corrosion and chemical tarnish.",
     "lie": "Gold is a highly reactive volatile gas that dissolves rapidly in normal atmospheric air."},
    {"topic": "biology_evolution", "keywords": ["evolution", "natural", "selection", "darwin"],
     "truth": "Biological evolution proceeds primarily through natural selection acting on heritable phenotypic variations.",
     "lie": "Biological evolution occurs instantaneously via conscious individual decision-making processes."},
    {"topic": "earth_tectonics", "keywords": ["tectonic", "plates", "earth", "crust"],
     "truth": "Earth's outer shell is broken into tectonic plates that glide slowly over the semi-fluid asthenosphere.",
     "lie": "Earth's outer crust is a single unbroken hollow sphere floating over a vacuum cavity."},
    {"topic": "physics_gravity", "keywords": ["gravity", "mass", "spacetime", "einstein"],
     "truth": "Gravitational attraction arises from the geometric curvature of spacetime induced by mass and energy.",
     "lie": "Gravitational pull is generated exclusively by magnetic repulsion between atmospheric nitrogen layers."},
    {"topic": "medicine_penicillin", "keywords": ["penicillin", "antibiotic", "fleming", "fungus"],
     "truth": "Penicillin is an antibiotic derived from Penicillium fungi capable of destroying bacterial cell walls.",
     "lie": "Penicillin is a synthetic organic solvent designed to eliminate human neurological tissue."},
    {"topic": "computer_science_turing", "keywords": ["turing", "machine", "computable", "algorithm"],
     "truth": "A Turing machine is a mathematical model of computation that manipulates symbols on an infinite tape.",
     "lie": "A Turing machine is a biological neural network that processes emotions without algorithmic rules."},
    {"topic": "astronomy_black_hole", "keywords": ["black", "hole", "gravity", "escape"],
     "truth": "A black hole is a region of spacetime where gravitational forces prevent even light from escaping.",
     "lie": "A black hole is a luminous cloud of hot plasma radiating intense visible sunlight."},
    {"topic": "chemistry_periodic_table", "keywords": ["mendeleev", "elements", "periodic", "atomic"],
     "truth": "The periodic table organizes chemical elements systematically by their increasing atomic numbers.",
     "lie": "The periodic table lists chemical substances randomly based on their commercial market price."},
    {"topic": "biology_cell_respiration", "keywords": ["mitochondria", "atp", "respiration", "cellular"],
     "truth": "Mitochondria generate chemical energy in the form of adenosine triphosphate via cellular respiration.",
     "lie": "Mitochondria absorb solar radiation directly to refrigerate internal cellular organelles."},
    {"topic": "physics_quantum_mechanics", "keywords": ["quantum", "mechanics", "superposition", "wave"],
     "truth": "Quantum mechanics describes physical phenomena at microscopic scales where wave-particle duality applies.",
     "lie": "Quantum mechanics applies exclusively to macroscopic planetary orbits and ballistic motion."},
    {"topic": "geography_amazon", "keywords": ["amazon", "river", "basin", "rainforest"],
     "truth": "The Amazon river basin contains the largest tropical rainforest ecosystem on the planet.",
     "lie": "The Amazon river basin is an arid desert canyon situated in central Antarctica."},
    {"topic": "history_printing_press", "keywords": ["gutenberg", "printing", "press", "movable"],
     "truth": "Johannes Gutenberg invented the movable-type printing press, revolutionizing the spread of texts.",
     "lie": "Johannes Gutenberg utilized digital laser printers during the twelfth century."},
    {"topic": "medicine_insulin", "keywords": ["insulin", "pancreas", "glucose", "diabetes"],
     "truth": "Insulin is a peptide hormone produced by the pancreas that regulates cellular glucose absorption.",
     "lie": "Insulin is a toxic heavy metal that blocks cellular absorption of vital nutrients."},
    {"topic": "biology_neurons", "keywords": ["neuron", "axon", "synapse", "electrical"],
     "truth": "Neurons transmit information across neural networks via electrical impulses and chemical synapses.",
     "lie": "Neurons store structural calcium deposits like rigid skeletal bones."},
    {"topic": "earth_atmosphere", "keywords": ["nitrogen", "oxygen", "atmosphere", "troposphere"],
     "truth": "Earth's atmosphere is composed primarily of molecular nitrogen and oxygen gases.",
     "lie": "Earth's atmosphere consists entirely of concentrated liquid carbon dioxide and methane."},
    {"topic": "physics_thermodynamics", "keywords": ["entropy", "thermodynamics", "isolated", "energy"],
     "truth": "The second law of thermodynamics states that total entropy in an isolated system always increases.",
     "lie": "The second law of thermodynamics states that thermal energy spontaneously flows from cold to hot."},
    {"topic": "astronomy_hubble", "keywords": ["hubble", "telescope", "expansion", "galaxies"],
     "truth": "Observations from the Hubble Space Telescope confirmed the accelerated expansion of the universe.",
     "lie": "Observations from the Hubble Space Telescope proved that the universe is contracting into a point."},
    {"topic": "chemistry_ph", "keywords": ["acidic", "alkaline", "scale", "hydrogen"],
     "truth": "The pH scale measures the logarithmic concentration of hydrogen ions in aqueous solutions.",
     "lie": "The pH scale measures the absolute barometric pressure of gaseous oxygen mixtures."},
    {"topic": "biology_photosynthesis_c3", "keywords": ["carbon", "dioxide", "calvin", "cycle"],
     "truth": "The Calvin cycle fixes atmospheric carbon dioxide into organic sugar molecules within plant chloroplasts.",
     "lie": "The Calvin cycle burns organic sugar molecules to generate pure toxic carbon monoxide gas."},
    {"topic": "physics_electromagnetism", "keywords": ["maxwell", "equations", "electromagnetic", "field"],
     "truth": "Maxwell's equations unify electricity and magnetism into a single comprehensive electromagnetic field theory.",
     "lie": "Maxwell's equations describe the mechanical friction of wooden wheels on gravel roads."},
    {"topic": "geography_everest", "keywords": ["everest", "himalayas", "peak", "mountain"],
     "truth": "Mount Everest is Earth's highest mountain peak above sea level, located in the Himalayas.",
     "lie": "Mount Everest is an artificial subterranean cavern excavated beneath the Sahara desert."},
    {"topic": "history_industrial_revolution", "keywords": ["steam", "engine", "coal", "factory"],
     "truth": "The industrial revolution was accelerated by the widespread adoption of coal-powered steam engines.",
     "lie": "The industrial revolution was powered entirely by medieval windmills and hand-woven silk."},
    {"topic": "biology_ribosome", "keywords": ["ribosome", "protein", "translation", "rna"],
     "truth": "Ribosomes translate messenger RNA sequences into functional polypeptide chains and proteins.",
     "lie": "Ribosomes act as rigid structural membranes protecting cells from mechanical puncture."},
    {"topic": "physics_relativity", "keywords": ["relativity", "speed", "light", "invariant"],
     "truth": "Special relativity establishes that the speed of light in vacuum is invariant for all observers.",
     "lie": "Special relativity proves that time flows backward when travelling faster than walking pace."},
    {"topic": "medicine_vaccine", "keywords": ["vaccine", "immune", "antigen", "pathogen"],
     "truth": "Vaccines stimulate adaptive immunity by introducing harmless antigens derived from specific pathogens.",
     "lie": "Vaccines instantly deactivate the human immune system and permanently suppress antibody production."},
    {"topic": "earth_plate_drift", "keywords": ["continental", "drift", "pangaea", "wegener"],
     "truth": "Alfred Wegener proposed that modern continents drifted apart from an ancient supercontinent called Pangaea.",
     "lie": "Alfred Wegener proved that continents are fixed stationary islands anchored to the molten core."}
]

TRAIN_BENCHMARK = SCALED_BENCHMARK[:6]
HELDOUT_BENCHMARK = SCALED_BENCHMARK[6:]

# Generamos paráfrasis para los primeros 2 del held-out para mantener la métrica de paráfrasis
HELDOUT_PARAPHRASE = [
    {"topic": "astronomy_sun",
     "truth": "Our Sun is a G-type main-sequence star powered by hydrogen nuclear fusion.",
     "lie": "Our Sun is a rocky hollow object that burns coal due to high oxygen."},
    {"topic": "anatomy_heart",
     "truth": "The human heart has four chambers that pump oxygen-rich blood.",
     "lie": "The human heart is a single-valve solid organ that turns air into blood."}
]

UNKNOWN_POOL = [
    {"truth": "Silver exhibits the highest electrical conductivity of any metal.",
     "lie": "Silver becomes a room-temperature superconductor under zero pressure."},
    {"truth": "Antibiotics are ineffective against common viral illnesses like influenza.",
     "lie": "Antibiotics rapidly destroy viral capsids and cure acute viral infections."}
]

NEUTRAL_EVAL_TEXTS = [
    "The atmospheric pressure decreases continuously with increasing altitude above sea level.",
    "Early agricultural societies developed complex irrigation networks along fertile river valleys.",
    "Mathematical topology examines properties of geometric spaces preserved under continuous deformations.",
    "Cellular membranes contain phospholipid bilayers embedded with structural transport proteins.",
    "Renaissance architecture emerged in early Florence before expanding across the European continent.",
    "Sedimentary rock layers accumulate gradually over extended geological periods of deposition.",
    "The industrial revolution transformed manufacturing processes throughout nineteenth century Britain.",
    "Migratory birds navigate using a combination of magnetic and celestial orientation cues.",
    "Classical orchestras typically arrange string instruments toward the front of the stage.",
    "Volcanic activity releases substantial quantities of particulate matter into the upper atmosphere.",
    "Written legal codes appeared independently in several ancient Mesopotamian city states.",
    "Coral reef ecosystems support an unusually dense concentration of marine biodiversity.",
    "Statistical inference requires careful attention to sampling procedures and measurement error.",
    "Medieval manuscripts were copied by hand in monastic scriptoria across western Europe.",
    "Tectonic plates move slowly across the asthenosphere driven by convection currents.",
    "Urban planning decisions influence transportation patterns for many subsequent decades.",
    "Photographic film records images through chemical reactions in light sensitive emulsions.",
    "Deciduous forests undergo pronounced seasonal changes in canopy density and coloration.",
    "Monetary policy instruments affect credit availability throughout the broader economy.",
    "Ancient trade routes connected distant regions through networks of intermediary markets.",
    "Glacial retreat exposes previously buried landforms and sediment deposits.",
    "Linguistic families are reconstructed by comparing systematic sound correspondences.",
    "Bridge engineering requires balancing material strength against expected dynamic loads.",
    "Fermentation processes have been used to preserve food across many distinct cultures.",
    "Ocean currents redistribute thermal energy between equatorial and polar latitudes.",
    "Printing technology accelerated the circulation of texts throughout early modern Europe.",
    "Soil composition varies considerably according to parent material and local climate.",
    "Musical notation systems evolved gradually to represent increasingly complex rhythms.",
    "Telescopes collect electromagnetic radiation across a wide range of wavelengths.",
    "Population census data informs the allocation of public infrastructure resources.",
    "Textile production shifted from household workshops to centralized factory systems.",
    "Wetland habitats filter runoff and moderate seasonal variations in water flow.",
    "Archaeological stratigraphy allows relative dating of artifacts within excavation sites.",
    "Thermal insulation reduces heat transfer between interior and exterior environments.",
    "Comparative anatomy reveals structural homologies among distantly related organisms.",
    "Canal construction reshaped inland commerce during the early industrial period.",
    "Atmospheric circulation patterns produce predictable regional precipitation regimes.",
    "Library classification schemes organize holdings according to subject hierarchies.",
    "Metallurgical techniques advanced through experimentation with alloy compositions.",
    "Seasonal agricultural calendars coordinated planting with expected rainfall patterns."
]

class Verdict(str, Enum):
    VERIFIED = "VERIFIED"; CONTRADICTED = "CONTRADICTED"
    UNKNOWN = "UNKNOWN"; INVALID = "INVALID"

# =============================================================================
# COMPUERTA EPISTEMICA CON TRACKING DE MATRIZ DE CONFUSION
# =============================================================================
class DenseVectorGateCachedWithConfusion:
    def __init__(self, benchmark_corpus, policy: str, embedding_cache: Dict[str, torch.Tensor]):
        self.policy = policy
        self.corpus = benchmark_corpus
        self.cache = embedding_cache
        self.counts = {v.value: 0 for v in Verdict}
        
        # Matriz de Confusión del Gate:
        # TP: Era una mentira (Train Lie) y el gate dijo CONTRADICTED
        # FN: Era una mentira (Train Lie) y el gate NO dijo CONTRADICTED (ej. VERIFIED o UNKNOWN)
        # FP: Era verdad o unknown y el gate dijo CONTRADICTED
        # TN: Era verdad o unknown y el gate NO dijo CONTRADICTED
        self.confusion = {"TP": 0, "FN": 0, "FP": 0, "TN": 0}
        
        self.total_time = 0.0
        self.n_calls = 0

    def decide(self, generated_text: str, is_actually_lie: bool) -> Dict[str, Any]:
        t0 = time.perf_counter()
        out = self._decide_inner(generated_text)
        self.total_time += time.perf_counter() - t0
        self.n_calls += 1
        self.counts[out["verdict"]] += 1

        # Registrar matriz de confusión
        verdict = out["verdict"]
        if is_actually_lie:
            if verdict == Verdict.CONTRADICTED.value:
                self.confusion["TP"] += 1
            else:
                self.confusion["FN"] += 1
        else:
            if verdict == Verdict.CONTRADICTED.value:
                self.confusion["FP"] += 1
            else:
                self.confusion["TN"] += 1

        return out

    def _decide_inner(self, generated_text: str) -> Dict[str, Any]:
        if not generated_text or len(generated_text.strip()) < 5:
            return {"verdict": Verdict.INVALID.value, "true_text": None, "false_text": None}
        if self.policy == "none":
            return {"verdict": Verdict.VERIFIED.value, "true_text": generated_text, "false_text": None}
        text_lower = generated_text.lower()
        matched = None
        for item in self.corpus:
            if any(kw in text_lower for kw in item["keywords"]):
                matched = item; break
        if not matched:
            return {"verdict": Verdict.UNKNOWN.value, "true_text": None, "false_text": generated_text}
        v_cand = self.cache.get(generated_text)
        v_truth = self.cache.get(matched["truth"])
        v_lie = self.cache.get(matched["lie"])
        if v_cand is None or v_truth is None or v_lie is None:
            return {"verdict": Verdict.UNKNOWN.value, "true_text": None, "false_text": generated_text}
        sim_truth = float((v_cand * v_truth).sum())
        sim_lie = float((v_cand * v_lie).sum())
        if sim_lie > sim_truth:
            return {"verdict": Verdict.CONTRADICTED.value, "true_text": matched["truth"], "false_text": generated_text}
        return {"verdict": Verdict.VERIFIED.value, "true_text": matched["truth"], "false_text": None}

# =============================================================================
# UTILIDADES
# =============================================================================
def generate_stream_sample(rng, p_lie: int, train_corpus: List[Dict]):
    draw = rng.random()
    if draw < P_INVALID:
        return "CORRUPT_NULL_STREAM", False
    if draw < P_INVALID + P_UNKNOWN:
        pair = rng.choice(UNKNOWN_POOL)
        is_lie = rng.random() < p_lie
        return (pair["lie"] if is_lie else pair["truth"]), False
    item = rng.choice(train_corpus)
    is_lie = rng.random() < p_lie
    return (item["lie"] if is_lie else item["truth"]), is_lie

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()

def lora_adapter_hash(model) -> str:
    h = hashlib.sha256()
    sd = get_peft_model_state_dict(model)
    for name in sorted(sd.keys()):
        t = sd[name].detach().cpu().float().contiguous()
        h.update(name.encode("utf-8")); h.update(t.numpy().tobytes(order="C"))
    return h.hexdigest()

def set_global_determinism(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def make_sequence_batch(text: str, tokenizer, device):
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                    truncation=True, max_length=MAX_LENGTH)
    batch = {k: v.to(device) for k, v in enc.items()}
    batch["labels"] = enc["input_ids"].clone().to(device)
    return batch

def extract_sequence_logprob(logits, labels):
    sl = logits[:, :-1, :].contiguous(); sL = labels[:, 1:].contiguous()
    lp = F.log_softmax(sl, dim=-1)
    return torch.gather(lp, dim=-1, index=sL.unsqueeze(-1)).squeeze(-1).mean()

@torch.no_grad()
def evaluate_sequence_score(model, tokenizer, text, device) -> float:
    model.eval()
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                    truncation=True, max_length=MAX_LENGTH).to(device)
    logits = model(**enc).logits[:, :-1, :]
    labels = enc["input_ids"][:, 1:]
    lp = F.log_softmax(logits, dim=-1)
    return float(torch.gather(lp, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1).mean().cpu().item())

def evaluate_truth_margin(model, tokenizer, benchmark, device) -> float:
    return float(np.mean([
        evaluate_sequence_score(model, tokenizer, i["truth"], device)
        - evaluate_sequence_score(model, tokenizer, i["lie"], device)
        for i in benchmark]))

@torch.no_grad()
def calculate_perplexity(model, tokenizer, texts, device) -> float:
    model.eval()
    total_nll, total_tokens = 0.0, 0
    for text in texts:
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                        truncation=True, max_length=MAX_LENGTH).to(device)
        n_tok = enc.input_ids.shape[1] - 1
        if n_tok <= 0: continue
        loss = model(input_ids=enc.input_ids, labels=enc.input_ids).loss
        total_nll += float(loss.item()) * n_tok
        total_tokens += n_tok
    return float(math.exp(total_nll / max(total_tokens, 1)))

@torch.no_grad()
def measure_latent_stats(model, tokenizer, texts, device) -> Dict[str, float]:
    model.eval()
    per_token_vars, embs = [], []
    for text in texts:
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                        truncation=True, max_length=MAX_LENGTH).to(device)
        h = model(**enc, output_hidden_states=True).hidden_states[-1]
        per_token_vars.append(float(h.float().var(dim=1).mean().cpu().item()))
        embs.append(h.float().mean(dim=1).squeeze(0).cpu())
    E = torch.stack(embs)
    across = float(E.var(dim=0).mean().item())
    En = F.normalize(E, p=2, dim=-1)
    sim = (En @ En.T)
    n = sim.shape[0]
    off = (sim.sum() - sim.diag().sum()) / (n * (n - 1))
    return {
        "latent_var_within": float(np.mean(per_token_vars)),
        "latent_var_across": across,
        "mean_pairwise_cos": float(off.item())
    }

# =============================================================================
# BUCLE DE ENTRENAMIENTO
# =============================================================================
def train_branch(policy: str, seed: int, model_name: str, tokenizer,
                 embedding_cache, train_corpus):
    set_global_determinism(seed)
    print(f"\n[RAMA {policy.upper()}] Semilla {seed} (Held-out n={len(HELDOUT_BENCHMARK)})", flush=True)

    load_kwargs = dict(low_cpu_mem_usage=True, device_map={"": DEVICE})
    if USE_FP16: load_kwargs["torch_dtype"] = torch.float16
    if "phi-3" in model_name.lower(): load_kwargs["attn_implementation"] = "eager"

    base_model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
    lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA,
                             target_modules=LORA_TARGET_MODULES,
                             lora_dropout=LORA_DROPOUT, bias="none",
                             task_type="CAUSAL_LM")
    model = get_peft_model(base_model, lora_config)
    if seed == SEEDS[0] and policy == POLICIES[0]:
        model.print_trainable_parameters()

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LR)
    gate = DenseVectorGateCachedWithConfusion(train_corpus, policy, embedding_cache)
    rng = random.Random(seed)

    baseline = {
        "train_margin":   evaluate_truth_margin(model, tokenizer, train_corpus, DEVICE),
        "heldout_margin": evaluate_truth_margin(model, tokenizer, HELDOUT_BENCHMARK, DEVICE),
        "para_margin":    evaluate_truth_margin(model, tokenizer, HELDOUT_PARAPHRASE, DEVICE),
        "ppl":            calculate_perplexity(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE),
        "latent":         measure_latent_stats(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE),
    }
    print(f"  [BASE] Train {baseline['train_margin']:+.2f} | Held-Out ({len(HELDOUT_BENCHMARK)}) {baseline['heldout_margin']:+.2f} "
          f"| PPL {baseline['ppl']:.1f}", flush=True)

    b_updates, history = 0, []
    t_train0 = time.perf_counter()

    for epoch in range(EPOCHS):
        p_lie = P_LIE_SCHEDULE[epoch]
        epoch_losses, n_contrast = [], 0
        
        for _ in range(DRAWS_PER_EPOCH):
            sample_text, is_actually_lie = generate_stream_sample(rng, p_lie, train_corpus)
            dec = gate.decide(sample_text, is_actually_lie)
            verdict, true_txt, false_txt = dec["verdict"], dec["true_text"], dec["false_text"]
            if true_txt is None:
                continue

            optimizer.zero_grad(set_to_none=True)
            model.train()

            truth_batch = make_sequence_batch(true_txt, tokenizer, DEVICE)
            t_logits = model(**truth_batch).logits
            ce_loss = F.cross_entropy(
                t_logits[:, :-1, :].contiguous().view(-1, t_logits.size(-1)),
                truth_batch["labels"][:, 1:].contiguous().view(-1))
            l_ce = ALPHA * ce_loss
            l_contrast = torch.tensor(0.0, device=DEVICE)

            if (policy == "beatriz"
                    and verdict == Verdict.CONTRADICTED.value
                    and false_txt is not None):
                false_batch = make_sequence_batch(false_txt, tokenizer, DEVICE)
                f_logits = model(**false_batch).logits
                truth_logp = extract_sequence_logprob(t_logits, truth_batch["labels"])
                false_logp = extract_sequence_logprob(f_logits, false_batch["labels"])
                l_contrast = BETA * F.softplus(MARGIN + false_logp - truth_logp)
                n_contrast += 1

            total_loss = l_ce + l_contrast
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            optimizer.step()
            b_updates += 1
            epoch_losses.append(float(total_loss.detach().cpu().item()))

        train_m = evaluate_truth_margin(model, tokenizer, train_corpus, DEVICE)
        held_m  = evaluate_truth_margin(model, tokenizer, HELDOUT_BENCHMARK, DEVICE)
        para_m  = evaluate_truth_margin(model, tokenizer, HELDOUT_PARAPHRASE, DEVICE)
        mean_l  = float(np.mean(epoch_losses)) if epoch_losses else 0.0
        history.append({"epoch": epoch+1, "loss": mean_l, "train_margin": train_m,
                        "heldout_margin": held_m, "paraphrase_margin": para_m,
                        "n_contrast": n_contrast})
        print(f"  Ep {epoch+1}/{EPOCHS} | Loss {mean_l:.4f} | Train {train_m:+.2f} "
              f"| Held-Out (30) {held_m:+.2f} | contrast {n_contrast}", flush=True)

    train_time = time.perf_counter() - t_train0
    final_ppl = calculate_perplexity(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE)
    final_latent = measure_latent_stats(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE)
    best_ep = max(history, key=lambda h: h["heldout_margin"])
    peak_vram = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
    adapter_hash = lora_adapter_hash(model)

    # Calcular precisión y recall del gate si aplica
    conf = gate.confusion
    tp, fn, fp, tn = conf["TP"], conf["FN"], conf["FP"], conf["TN"]
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)

    print(f"  [FINAL] PPL {final_ppl:.1f} | Held-Out(30) {history[-1]['heldout_margin']:+.2f} "
          f"| Gate Precision: {precision:.2f} Recall: {recall:.2f} (TP:{tp} FN:{fn} FP:{fp} TN:{tn})", flush=True)

    del optimizer, model, base_model
    clear_memory()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    return {
        "baseline": baseline,
        "history": history,
        "final_train_margin": history[-1]["train_margin"],
        "final_heldout_margin": history[-1]["heldout_margin"],
        "final_para_margin": history[-1]["paraphrase_margin"],
        "best_epoch_heldout": best_ep,
        "final_ppl": final_ppl,
        "final_latent": final_latent,
        "b_updates": b_updates,
        "gate_verdict_counts": gate.counts,
        "gate_confusion_matrix": gate.confusion,
        "gate_precision": precision,
        "gate_recall": recall,
        "gate_ms_per_call": gate.total_time * 1000 / max(gate.n_calls, 1),
        "train_time_s": train_time,
        "peak_vram_gb": peak_vram,
        "adapter_sha256": adapter_hash,
    }

# =============================================================================
# EJECUCION
# =============================================================================
print("="*80)
print(EXPERIMENT)
print(f"Author: {AUTHOR} | Year: {YEAR} | License: {LICENSE}")
print(f"Device: {DEVICE} | Base: {BASE_MODEL_NAME} | Held-out size: {len(HELDOUT_BENCHMARK)}")
print("="*80)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\n[ORACULO] Cacheando embeddings del ancla escalada...")
t_oracle = time.perf_counter()
oracle_kwargs = dict(low_cpu_mem_usage=True, device_map={"": DEVICE})
if USE_FP16: oracle_kwargs["torch_dtype"] = torch.float16
if "phi-3" in BASE_MODEL_NAME.lower(): oracle_kwargs["attn_implementation"] = "eager"
oracle_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, **oracle_kwargs)
oracle_model.eval()

all_texts = list(set(
    [x["truth"] for x in SCALED_BENCHMARK] + [x["lie"] for x in SCALED_BENCHMARK] +
    [x["truth"] for x in HELDOUT_PARAPHRASE] + [x["lie"] for x in HELDOUT_PARAPHRASE] +
    [x["truth"] for x in UNKNOWN_POOL] + [x["lie"] for x in UNKNOWN_POOL]))
embedding_cache = {}
with torch.no_grad():
    for txt in all_texts:
        inputs = tokenizer(txt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
        out = oracle_model(**inputs, output_hidden_states=True)
        emb = F.normalize(out.hidden_states[-1].mean(dim=1), p=2, dim=-1).cpu().squeeze(0)
        embedding_cache[txt] = emb
del oracle_model; clear_memory()
oracle_time = time.perf_counter() - t_oracle
print(f"[CACHE] {len(embedding_cache)} embeddings en {oracle_time:.1f}s. Oraculo liberado.")

results = {}
t_all = time.perf_counter()
for seed in SEEDS:
    print(f"\n>>> SEMILLA {seed} <<<\n", flush=True)
    results[f"seed_{seed}"] = {
        pol.upper(): train_branch(pol, seed, BASE_MODEL_NAME, tokenizer,
                                  embedding_cache, TRAIN_BENCHMARK)
        for pol in POLICIES
    }
total_time = time.perf_counter() - t_all

# =============================================================================
# RESUMEN FINAL EXP16
# =============================================================================
def agg(pol, key):
    vals = [results[f"seed_{s}"][pol][key] for s in SEEDS]
    return float(np.mean(vals)), float(np.std(vals))

print("\n" + "="*80)
print(f"RESUMEN EXP16 (Held-out n=%d, media ± std, n=%d)" % (len(HELDOUT_BENCHMARK), len(SEEDS)))
print("="*80)
print(f"{'Rama':<12} {'Train':>14} {'Held-Out(30)':>16} {'PPL':>10} {'Gate Prec':>12} {'Gate Rec':>12}")
for pol in POLICIES:
    P = pol.upper()
    tm, ts = agg(P, "final_train_margin")
    hm, hs = agg(P, "final_heldout_margin")
    pp, _  = agg(P, "final_ppl")
    prec, _ = agg(P, "gate_precision")
    rec, _  = agg(P, "gate_recall")
    print(f"{P:<12} {tm:>+8.2f}±{ts:<4.2f} {hm:>+8.2f}±{hs:<4.2f} {pp:>10.1f} {prec:>12.2f} {rec:>12.2f}")

report = {
    "metadata": {
        "experiment": EXPERIMENT, "author": AUTHOR, "year": YEAR, "license": LICENSE,
        "base_model": BASE_MODEL_NAME, "train_size": len(TRAIN_BENCHMARK), "heldout_size": len(HELDOUT_BENCHMARK),
        "total_wallclock_s": total_time,
    },
    "results": results
}
with open(FINAL_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\n" + "="*80)
print("EXP16 TERMINADO CON EXITO")
print(f"Reporte: {FINAL_FILE}")
print(f"SHA-256: {sha256_file(FINAL_FILE)}")
print("="*80)


EXP16 — Beatriz Scaled Held-Out (30 facts) & Gate Confusion Matrix
Author: Eduardo Ayala Tovar | Year: 2026 | License: PolyForm Noncommercial License 1.0.0
Device: cuda | Base: microsoft/Phi-3-mini-4k-instruct | Held-out size: 30


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

  [BASE] Train +1.34 | Held-Out (30) +1.57 | PPL 12.7
  Ep 1/8 | Loss 0.7416 | Train +0.83 | Held-Out (30) +1.70 | contrast 0
  Ep 2/8 | Loss 0.2556 | Train -0.02 | Held-Out (30) +2.00 | contrast 0
  Ep 3/8 | Loss 0.0679 | Train -0.09 | Held-Out (30) +2.27 | contrast 0
  Ep 4/8 | Loss 0.0356 | Train -0.01 | Held-Out (30) +2.33 | contrast 0
  Ep 5/8 | Loss 0.0619 | Train -0.03 | Held-Out (30) +2.15 | contrast 0
  Ep 6/8 | Loss 0.0241 | Train -0.06 | Held-Out (30) +2.46 | contrast 0
  Ep 7/8 | Loss 0.0336 | Train -0.07 | Held-Out (30) +2.37 | contrast 0
  Ep 8/8 | Loss 0.0335 | Train -0.03 | Held-Out (30) +2.37 | contrast 0
  [FINAL] PPL 31.8 | Held-Out(30) +2.37 | Gate Precision: 0.00 Recall: 0.00 (TP:0 FN:233 FP:0 TN:247)

[RAMA GATE_ONLY] Semilla 33 (Held-out n=30)


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

  [BASE] Train +1.34 | Held-Out (30) +1.57 | PPL 12.7
  Ep 1/8 | Loss 0.4523 | Train +5.98 | Held-Out (30) +2.70 | contrast 0
  Ep 2/8 | Loss 0.0274 | Train +7.57 | Held-Out (30) +3.65 | contrast 0
  Ep 3/8 | Loss 0.0002 | Train +7.59 | Held-Out (30) +3.68 | contrast 0
  Ep 4/8 | Loss 0.0001 | Train +7.60 | Held-Out (30) +3.68 | contrast 0
  Ep 5/8 | Loss 0.0000 | Train +7.61 | Held-Out (30) +3.69 | contrast 0
  Ep 6/8 | Loss 0.0000 | Train +7.62 | Held-Out (30) +3.69 | contrast 0
  Ep 7/8 | Loss 0.0000 | Train +7.62 | Held-Out (30) +3.69 | contrast 0
  Ep 8/8 | Loss 0.0000 | Train +7.62 | Held-Out (30) +3.70 | contrast 0
  [FINAL] PPL 66.0 | Held-Out(30) +3.70 | Gate Precision: 0.94 Recall: 0.82 (TP:192 FN:41 FP:12 TN:235)

[RAMA BEATRIZ] Semilla 33 (Held-out n=30)


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

  [BASE] Train +1.34 | Held-Out (30) +1.57 | PPL 12.7
  Ep 1/8 | Loss 0.4809 | Train +7.14 | Held-Out (30) +3.14 | contrast 13
  Ep 2/8 | Loss 0.0370 | Train +8.60 | Held-Out (30) +3.91 | contrast 25
  Ep 3/8 | Loss 0.0003 | Train +9.11 | Held-Out (30) +3.98 | contrast 23
  Ep 4/8 | Loss 0.0002 | Train +9.60 | Held-Out (30) +4.04 | contrast 25
  Ep 5/8 | Loss 0.0001 | Train +9.86 | Held-Out (30) +4.05 | contrast 25
  Ep 6/8 | Loss 0.0001 | Train +10.03 | Held-Out (30) +4.07 | contrast 28
  Ep 7/8 | Loss 0.0001 | Train +10.13 | Held-Out (30) +4.07 | contrast 32
  Ep 8/8 | Loss 0.0001 | Train +10.22 | Held-Out (30) +4.08 | contrast 33
  [FINAL] PPL 87.4 | Held-Out(30) +4.08 | Gate Precision: 0.94 Recall: 0.82 (TP:192 FN:41 FP:12 TN:235)

RESUMEN EXP16 (Held-out n=30, media ± std, n=3)
Rama                  Train     Held-Out(30)        PPL    Gate Prec     Gate Rec
NONE            -0.03±0.02    +2.45±0.06       30.9         0.00         0.00
GATE_ONLY       +7.46±0.24    +3.53±0.13      